In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.cluster import DBSCAN
import folium

In [2]:
df = pd.read_csv('movement.csv')

cols = ['start_lat', 'start_lon', 'end_lat', 'end_lon']
df = df[cols].dropna().reset_index(drop=True)
df.head()

,start_lat,start_lon,end_lat,end_lon
0,55.658398,12.514628,55.658348,12.515684
1,55.658348,12.515684,55.659286,12.519309
2,55.659286,12.519309,55.677685,12.522237
3,55.677685,12.522237,55.676945,12.520396
4,55.676945,12.520396,55.655346,12.537441


In [3]:

start_pts = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df['start_lon'], df['start_lat']),
    crs='EPSG:4326'
).to_crs(epsg=25832)

df['start_X'] = start_pts.geometry.x
df['start_Y'] = start_pts.geometry.y

end_pts = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df['end_lon'], df['end_lat']),
    crs='EPSG:4326'
).to_crs(epsg=25832)

df['end_X'] = end_pts.geometry.x
df['end_Y'] = end_pts.geometry.y

print('Coordinates converted to metres')
df[['start_X', 'start_Y', 'end_X', 'end_Y']].head()

Coordinates converted to metres


,start_X,start_Y,end_X,end_Y
0,721078.781674,6.173663e+06,721145.458425,6.173661e+06
1,721145.458425,6.173661e+06,721368.078357,6.173777e+06
2,721368.078357,6.173777e+06,721448.159371,6.175832e+06
3,721448.159371,6.175832e+06,721336.647596,6.175744e+06
4,721336.647596,6.175744e+06,722530.439044,6.173396e+06


In [ ]:
EPS         = 300   
MIN_SAMPLES = 100


algo = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
algo.fit(df[['start_X', 'start_Y']])
df['start_cluster'] = pd.Series(algo.labels_, index=df.index)

algo = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
algo.fit(df[['end_X', 'end_Y']])
df['end_cluster'] = pd.Series(algo.labels_, index=df.index)

n_start = len(set(df['start_cluster'])) - (1 if -1 in df['start_cluster'].values else 0)
n_end   = len(set(df['end_cluster']))   - (1 if -1 in df['end_cluster'].values else 0)
print(f'Start clusters: {n_start}')
print(f'End clusters  : {n_end}')